# Full Profiling

[`util:profile()`]({docs}/functions/util/profile) is the all-in-one profiling function. It executes a query and returns everything: the result, timing, memory, the expression tree, and profiler statistics — all in a single map.

## Basic Profiling

In [ ]:
let $p := util:profile('
    for $i in 1 to 1000
    let $sq := $i * $i
    where $sq mod 7 = 0
    return $sq
')
return map {
    "item-count": count($p?result),
    "time": string($p?time),
    "memory-bytes": $p?memory
}

## The Full Result Map

[`util:profile()`]({docs}/functions/util/profile) returns a map with these keys:

| Key | Type | Description |
|-----|------|-------------|
| `result` | `item()*` | The actual query result |
| `time` | `xs:dayTimeDuration` | Execution time |
| `memory` | `xs:integer` | Memory delta in bytes |
| `plan` | `element(explain)` | Compiled expression tree (same as [`util:explain()`]({docs}/functions/util/explain)) |
| `stats` | `element()` | Profiler statistics XML (function calls, index usage) |

## Viewing the Expression Plan

The `plan` key contains the same output as [`util:explain()`]({docs}/functions/util/explain), but for the query that was actually executed:

In [ ]:
let $p := util:profile('
    for $x in 1 to 10
    where $x > 5
    return $x * 2
')
return $p?plan

## Viewing Profiler Statistics

The `stats` key contains XML from eXist's built-in profiler — function call counts, index usage, and optimization data:

In [ ]:
let $p := util:profile('
    let $nums := 1 to 10000
    return (count($nums), sum($nums), max($nums))
')
return $p?stats

## Per-Query Isolation

Each [`util:profile()`]({docs}/functions/util/profile) call collects stats independently. Even under concurrent load, the profiling data reflects only the profiled query — not other queries running at the same time:

In [ ]:
let $p1 := util:profile('sum(1 to 10000)')
let $p2 := util:profile('count(1 to 10000)')
return map {
    "sum-time": string($p1?time),
    "count-time": string($p2?time),
    "sum-result": $p1?result,
    "count-result": $p2?result
}

## Index Usage Report

[`util:index-report()`]({docs}/functions/util/index-report) is a focused variant that shows just the index and optimization data:

In [ ]:
util:index-report('
    for $i in 1 to 100
    return $i * $i
')

For queries against indexed collections, this shows which indexes were consulted:

In [ ]:
(: Run this against a collection with range indexes to see index usage :)
util:index-report('collection("/db/apps/demo/data")//author')

## Profiling a Database Query

Profile a real query against stored data:

In [ ]:
let $p := util:profile('
    collection("/db")//system:modules/system:module
')
return map {
    "modules-found": count($p?result),
    "time": string($p?time),
    "memory": $p?memory
}

## Performance Comparison Pattern

Compare two query approaches with full profiling:

In [ ]:
let $approach-a := util:profile('
    for $i in 1 to 5000
    where $i mod 2 = 0
    return $i
')
let $approach-b := util:profile('
    (1 to 5000)[. mod 2 = 0]
')
return map {
    "flwor-time": string($approach-a?time),
    "flwor-count": count($approach-a?result),
    "predicate-time": string($approach-b?time),
    "predicate-count": count($approach-b?result),
    "same-result": deep-equal(
        sort($approach-a?result),
        sort($approach-b?result)
    )
}

## Debugging Slow Queries

When a query is slow, use this pattern to find the bottleneck:

In [ ]:
(: Step 1: Profile the whole query :)
let $full := util:profile('
    let $data := 1 to 50000
    let $filtered := $data[. mod 7 = 0]
    let $mapped := for $x in $filtered return $x * $x
    return sum($mapped)
')

(: Step 2: Profile individual steps :)
let $step1 := util:track(1 to 50000, "generate")
let $step2 := util:track(
    $step1?value[. mod 7 = 0],
    "filter"
)
let $step3 := util:track(
    for $x in $step2?value return $x * $x,
    "map"
)
let $step4 := util:track(
    sum($step3?value),
    "sum"
)

return map {
    "total": string($full?time),
    "generate": string($step1?time),
    "filter": string($step2?time),
    "map": string($step3?time),
    "sum": string($step4?time),
    "result": $step4?value
}

## Function Reference

| Function | Signature | Returns |
|----------|-----------|---------|
| [`util:time`]({docs}/functions/util/time) | `($expr) as item()*` | Result unchanged; logs time |
| [`util:time`]({docs}/functions/util/time) | `($expr, $label) as item()*` | Same with custom label |
| [`util:memory`]({docs}/functions/util/memory) | `($expr) as item()*` | Result unchanged; logs memory |
| [`util:memory`]({docs}/functions/util/memory) | `($expr, $label) as item()*` | Same with custom label |
| [`util:track`]({docs}/functions/util/track) | `($expr) as map(*)` | `{ time, memory, value }` |
| [`util:track`]({docs}/functions/util/track) | `($expr, $label) as map(*)` | `{ time, memory, value, label }` |
| [`util:explain`]({docs}/functions/util/explain) | `($query) as element(explain)` | Expression tree XML |
| [`util:explain`]({docs}/functions/util/explain) | `($query, $path) as element(explain)` | Same with module path |
| [`util:profile`]({docs}/functions/util/profile) | `($query) as map(*)` | `{ result, time, memory, plan, stats }` |
| [`util:profile`]({docs}/functions/util/profile) | `($query, $path) as map(*)` | Same with module path |
| [`util:index-report`]({docs}/functions/util/index-report) | `($query) as element()` | Index usage XML |